This notebook loads the pickled results produced by `scaling_effdim_dloc.py` and plots how the normalized effective dimension (ED, Eq. 12 of the paper) scales with the local per-parameter basis dimension Dloc ($\tilde d$, `local_dim_param`), for a few fixed architectures (input-basis dimension D, no. of parameters M, tensor-train bond dimension $\chi$). Each curve is annotated with the correlation-spectrum purity $\mathrm{tr}\,S^4$ of the models it was computed from.

Import the plotting and numerical packages, and add the `useful_functions` folder to the module search path so the tensor-network, FIM, and model-construction helper modules used by the generating script can be (re)imported.

In [ ]:
# Importing necessary packages
import sys
import os
import importlib
import pickle
import numpy

import pennylane.numpy as np




import matplotlib.pyplot as plt


# Current path for importing custom functions
path_base = '/home/b/b309245/FIM_Training_Bias_RegressionModels/fourier_models_training_and_fim/'
sys.path.insert(0, path_base + 'useful_functions')

import model_constructor_functions
importlib.reload(model_constructor_functions)

import ortho_matrices_functions
importlib.reload(ortho_matrices_functions)

import tensor_network_functions_np
importlib.reload(tensor_network_functions_np)

import FIM_functions_jax
importlib.reload(FIM_functions_jax)

import training_functions_jax
importlib.reload(training_functions_jax)

Set the results folder and the fixed parameters shared by all experiments (M, tensor-train bond dimension $\chi$, dec_exp, no. of parameter samples/realizations), then list the input-basis dimensions D (`dim_in_vec`) identifying each experiment/curve whose per-Dloc pickled results are to be loaded.

In [ ]:
results_folder = '/work/bd1179/b309245/fourier_models_train_and_FIM/scaling_effdim_TN_Dloc/'

# No. of random parameter samples for evaluating normalized eff. dim.
no_samples = 200
no_par_samples_name = str(no_samples)

# No. of random orthogonal matrix realizations per S decay exponent
no_matrix_realiz = 30
no_V_samples_name = str(no_matrix_realiz)

# No. of parameters
no_params = 60
name_no_params = str(no_params)

# Vector of bond dimensions
bond_dim = 100
name_bond_dim = str(bond_dim)

# Decay exponent
dec_exp = 0.0
name_dec_exp = '0p0'

########################### Exps. to plot ###########################

# Dimension of input function space
dim_in_vec = [50, 90]
name_dim_in_vec = [str(dim_in) for dim_in in dim_in_vec]

no_exps = len(dim_in_vec)

For each experiment (each D value), find and load all pickled `norm_eff_dim_*` files matching that architecture, concatenating the swept Dloc values, correlation-spectrum purity $\mathrm{tr}\,S^4$, and normalized ED arrays across all random model realizations into a per-experiment dictionary `dict_exps[i]`.

In [ ]:
dict_exps = dict()

for i in range(no_exps):
    dim_in = dim_in_vec[i]
    name_dim_in = name_dim_in_vec[i]

    name_end_0 = ('_BondDim' + name_bond_dim + '_DimIn' + name_dim_in + '_Nparams' + name_no_params +
                  '_DecExp' + name_dec_exp  + '_NsamplesPar' + no_par_samples_name + 
                  '_NsamplesV' + no_V_samples_name + '_DimLocPar')
    
    namefileini = 'norm_eff_dim' + name_end_0
    listallfiles = [f for f in os.listdir(results_folder) if (f.startswith(namefileini))]
    print(len(listallfiles))
    
    loc_dim_all = []
    purity_S_all = []
    norm_eff_dim_all = []
    
    for filename in listallfiles:
        path_file = os.path.join(results_folder, filename)
        with open(path_file, 'rb') as f:
            dict_norm_ed = pickle.load(f)
    
        locdim_vec = dict_norm_ed['loc_dim_all']
        purity_S_vec = dict_norm_ed['purity_S_all']
        norm_eff_dim_vec = dict_norm_ed['norm_eff_dim_all']
    
        loc_dim_all.append(locdim_vec)
        purity_S_all.append(purity_S_vec)
        norm_eff_dim_all.append(norm_eff_dim_vec)
    
    loc_dim_all = np.concatenate(loc_dim_all)
    purity_S_all = np.concatenate(purity_S_all)
    norm_eff_dim_all = np.concatenate(norm_eff_dim_all)

    dict_exp = dict()
    dict_exp['dim_in'] = dim_in
    dict_exp['purity_S_all'] = purity_S_all
    dict_exp['loc_dim_all'] = loc_dim_all
    dict_exp['norm_eff_dim_all'] = norm_eff_dim_all
    dict_exps[i] = dict_exp

Compute the mean correlation-spectrum purity $\mathrm{tr}\,S^4$ for each experiment and plot the normalized effective dimension $\hat d_{\mathrm{eff}}$ vs Dloc ($\tilde d$), one curve per fixed (M, D) architecture, annotated with $\mathrm{tr}\,S^4$ in the legend.

In [ ]:
fs = 28
figsize = (8,4)

plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

clrs = ['tab:blue', 'tab:orange']
for i in range(no_exps):
    dict_exp = dict_exps[i]
    dim_in = dict_exp['dim_in']
    loc_dim_all = dict_exp['loc_dim_all']
    norm_eff_dim_all = dict_exp['norm_eff_dim_all']
    purity_S_all = numpy.asarray(dict_exp['purity_S_all'])
    purity_rounded = round(numpy.mean(purity_S_all), 2)
    print('S purity: ', purity_rounded, ' with std. ', np.std(purity_S_all))
    lbl = r'$M=' + str(no_params) + ',\,D=' + str(dim_in) + ',\,\mathrm{tr}\,S^4=' + str(purity_rounded) + '$'
    x = np.asarray(loc_dim_all)
    y = np.asarray(norm_eff_dim_all)
    ax.plot(x, y, '.', color=clrs[i], markersize=12, label=lbl)

ax.legend(fontsize=22, loc='lower left')
ax.set_xlabel(r'$\tilde{d}$', fontsize=fs)
ax.set_ylabel(r'$\hat{d}_{\mathrm{eff}}$', fontsize=fs)
#ax.set_xscale('log')
#ax.set_yscale('log')
ax.set_ylim([0.725, 0.86])

fig.savefig('EffDimScaling_vs_Dloc_TN__chi_100.png', bbox_inches='tight')
fig.savefig('EffDimScaling_vs_Dloc_TN__chi_100.pdf', bbox_inches='tight')